In [ ]:
!pip install mistralai -q


In [3]:
from google.colab import userdata

api_key = userdata.get("MISTRAL_API_KEY")

In [4]:
from mistralai.client import Mistral

client = Mistral(api_key=api_key)



In [5]:
!pip install -q langchain-mistralai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 1.1 MB/s eta 0:00:00


In [6]:
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
import requests

In [7]:
# create tool
@tool
def multiply(a:int, b :int )->int:
  """ given 2 numbers a and b this tool return their product"""
  return a*b

In [8]:
print(multiply.invoke({'a':3,'b':4}))

12


In [45]:
 multiply.name

'multiply'

In [46]:
multiply.args
multiply.description

'given 2 numbers a and b this tool return their product'

In [83]:
query= HumanMessage(content="what is 3 multiply 7")
messages=[query]

In [60]:
# tool binding
llm =ChatMistralAI(api_key=api_key)
llm_with_tools=llm.bind_tools([multiply])

In [84]:
result=llm_with_tools.invoke(messages)
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'BsXkvTfon', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a": 3, "b": 7}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 99, 'total_tokens': 116, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ee131-b3f5-7531-8db8-15a46aa019f5-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 7}, 'id': 'BsXkvTfon', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 17, 'total_tokens': 116})

In [85]:
messages.append(result)
messages


[HumanMessage(content='what is 3 multiply 7', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'BsXkvTfon', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a": 3, "b": 7}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 99, 'total_tokens': 116, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ee131-b3f5-7531-8db8-15a46aa019f5-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 7}, 'id': 'BsXkvTfon', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 17, 'total_tokens': 116})]

In [86]:
from langchain_core.messages import ToolMessage

tool_call_info = result.tool_calls[0]
tool_output = multiply.invoke(tool_call_info['args'])
tool_output
# message_from_tool = ToolMessage(content=str(tool_output), name=tool_call_info['name'], tool_call_id=tool_call_info['id'])

21

In [87]:
message_from_tool

ToolMessage(content='21', name='multiply', tool_call_id='QUvY98jga')

In [88]:
messages.append(message_from_tool)

In [80]:
messages

[HumanMessage(content='what is 3 multiply 7', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'csG7Db9Gz', 'type': 'function', 'function': {'name': 'multiply', 'arguments': '{"a": 3, "b": 7}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 99, 'total_tokens': 116, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small', 'model': 'mistral-small', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ee0d1-72f4-7c00-996f-5fe1abe59aba-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 7}, 'id': 'csG7Db9Gz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 17, 'total_tokens': 116}),
 ToolMessage(content='21', name='multiply', tool_call_id='P5jr74rHR'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'QUvY98jga', 'type': 'function', 'function': {'name': 'multiply', 'a

In [90]:
llm_with_tools.invoke([query, result, message_from_tool]).content

HTTPStatusError: Error response 400 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Unexpected tool call id QUvY98jga in tool results","type":"invalid_request_message_order","param":null,"code":"3230","raw_status_code":400}